# DQN Breakout — Production Pipeline (Colab Pro, Unattended)
**Member 2: Kelvin · ALE/Breakout-v5 · Stable Baselines3**



## 1 · Install (run once → then Restart session)

In [3]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" "ale-py" "autorom[accept-rom-license]" imageio imageio-ffmpeg
print("Installed. Runtime -> Restart session, then Runtime -> Run all.")

Installed. Runtime -> Restart session, then Runtime -> Run all.


## 2 · Environment checks + Drive mount

In [4]:
import torch
assert torch.cuda.is_available(), "STOP: Runtime -> Change runtime type -> GPU, then rerun"
print("GPU:", torch.cuda.get_device_name(0))

import ale_py, gymnasium as gym
gym.register_envs(ale_py)
gym.make("ALE/Breakout-v5")
print("ALE OK")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/dqn_breakout_run"
for sub in ["models", "logs", "flags", "artifacts"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
print("Drive workspace:", DRIVE_DIR)

GPU: NVIDIA A100-SXM4-80GB
ALE OK
Mounted at /content/drive
Mounted at /content/drive
Drive workspace: /content/drive/MyDrive/dqn_breakout_run
Drive workspace: /content/drive/MyDrive/dqn_breakout_run


## 3 · Write submission scripts (train.py / play.py) → saved to Drive too

In [5]:
%%writefile train.py
"""
train.py
========
Trains a DQN agent on an Atari environment (Breakout) using Stable Baselines3.

Team usage:
    Each member runs their 10 hyperparameter experiments by passing values
    on the command line, e.g.:

    python train.py --exp-name m2_exp01 --lr 1e-4 --gamma 0.99 \
        --batch-size 32 --eps-start 1.0 --eps-end 0.01 --eps-decay 0.10 \
        --timesteps 500000 --policy CnnPolicy

    Logs (reward trends, episode length) go to ./logs/<exp-name>/ and can be
    inspected with TensorBoard:  tensorboard --logdir logs

    The best model overall should be re-saved/copied as dqn_model.zip
    (the script already saves models/<exp-name>/dqn_model.zip).
"""

import argparse
import os

import ale_py                      # registers ALE/* Atari environments with Gymnasium
import gymnasium as gym
gym.register_envs(ale_py)          # belt-and-suspenders explicit registration

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor


ENV_ID = "ALE/Breakout-v5"


def parse_args():
    p = argparse.ArgumentParser(description="Train a DQN agent on Atari Breakout")
    p.add_argument("--exp-name", type=str, default="dqn_breakout",
                   help="Name of this experiment (used for log/model folders)")
    p.add_argument("--policy", type=str, default="CnnPolicy",
                   choices=["CnnPolicy", "MlpPolicy"],
                   help="Policy network architecture")
    p.add_argument("--lr", type=float, default=1e-4, help="Learning rate")
    p.add_argument("--gamma", type=float, default=0.99, help="Discount factor")
    p.add_argument("--batch-size", type=int, default=32,
                   help="Minibatch size sampled from the replay buffer")
    p.add_argument("--eps-start", type=float, default=1.0,
                   help="Initial epsilon for the epsilon-greedy policy")
    p.add_argument("--eps-end", type=float, default=0.01,
                   help="Final epsilon after decay")
    p.add_argument("--eps-decay", type=float, default=0.10,
                   help="Fraction of total timesteps over which epsilon decays "
                        "(SB3's exploration_fraction)")
    p.add_argument("--buffer-size", type=int, default=100_000,
                   help="Replay buffer size")
    p.add_argument("--timesteps", type=int, default=500_000,
                   help="Total training timesteps")
    p.add_argument("--seed", type=int, default=42, help="Random seed")
    return p.parse_args()


def main():
    args = parse_args()

    log_dir = os.path.join("logs", args.exp_name)
    model_dir = os.path.join("models", args.exp_name)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Environment
    # ------------------------------------------------------------------
    # make_atari_env applies the standard Atari wrappers:
    #   - NoopReset, frame skipping, 84x84 grayscale resize, reward clipping
    # VecFrameStack(4) stacks the last 4 frames so the CNN can infer motion
    # (ball direction/velocity in Breakout).
    env = make_atari_env(ENV_ID, n_envs=1, seed=args.seed)
    env = VecFrameStack(env, n_stack=4)

    # Separate evaluation environment (same wrappers) for unbiased eval
    eval_env = make_atari_env(ENV_ID, n_envs=1, seed=args.seed + 100)
    eval_env = VecFrameStack(eval_env, n_stack=4)

    # ------------------------------------------------------------------
    # Agent
    # ------------------------------------------------------------------
    model = DQN(
        policy=args.policy,
        env=env,
        learning_rate=args.lr,
        gamma=args.gamma,
        batch_size=args.batch_size,
        buffer_size=args.buffer_size,
        learning_starts=50_000,          # fill replay buffer before learning
        target_update_interval=10_000,   # target network sync frequency
        train_freq=4,                    # gradient step every 4 env steps
        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_decay,
        optimize_memory_usage=False,
        tensorboard_log=log_dir,
        verbose=1,
        seed=args.seed,
    )

    # ------------------------------------------------------------------
    # Callbacks: periodically evaluate and keep the best checkpoint.
    # EvalCallback also logs mean reward and mean episode length, which
    # covers the "reward trends / episode length" logging requirement.
    # ------------------------------------------------------------------
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=model_dir,
        log_path=log_dir,
        eval_freq=25_000,
        n_eval_episodes=5,
        deterministic=True,
        render=False,
    )

    # ------------------------------------------------------------------
    # Train
    # ------------------------------------------------------------------
    print(f"\n=== Experiment: {args.exp_name} ===")
    print(f"policy={args.policy} lr={args.lr} gamma={args.gamma} "
          f"batch={args.batch_size} eps=({args.eps_start}->{args.eps_end} "
          f"over {args.eps_decay*100:.0f}% of steps) "
          f"timesteps={args.timesteps}\n")

    model.learn(
        total_timesteps=args.timesteps,
        callback=eval_callback,
        log_interval=100,
        progress_bar=True,
    )

    # ------------------------------------------------------------------
    # Save final model as dqn_model.zip (per assignment requirement)
    # ------------------------------------------------------------------
    save_path = os.path.join(model_dir, "dqn_model.zip")
    model.save(save_path)
    print(f"\nModel saved to {save_path}")
    print(f"Best-eval checkpoint saved to {model_dir}/best_model.zip")
    print(f"TensorBoard logs in {log_dir} (run: tensorboard --logdir logs)")


if __name__ == "__main__":
    main()


Writing train.py
Writing train.py


In [6]:
%%writefile play.py
"""
play.py
=======
Loads the trained DQN model and plays Atari Breakout with a greedy policy.

Usage:
    python play.py --model models/best/dqn_model.zip --episodes 5

Notes:
- Uses the SAME preprocessing as train.py (Atari wrappers + 4-frame stack),
  which is required for the model's CNN input to match.
- Greedy evaluation: model.predict(..., deterministic=True) always picks
  argmax Q(s, a) — this is SB3's equivalent of a GreedyQPolicy.
- Rendering: render_mode="human" opens a GUI window showing the game live.
"""

import argparse

import ale_py                      # registers ALE/* Atari environments with Gymnasium
import gymnasium as gym
gym.register_envs(ale_py)          # belt-and-suspenders explicit registration

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack


ENV_ID = "ALE/Breakout-v5"


def parse_args():
    p = argparse.ArgumentParser(description="Play Breakout with a trained DQN agent")
    p.add_argument("--model", type=str, default="dqn_model.zip",
                   help="Path to the trained model zip")
    p.add_argument("--episodes", type=int, default=5,
                   help="Number of episodes to play")
    return p.parse_args()


def main():
    args = parse_args()

    # Same environment + wrappers as training, but rendered to screen.
    env = make_atari_env(
        ENV_ID,
        n_envs=1,
        seed=0,
        env_kwargs={"render_mode": "human"},
    )
    env = VecFrameStack(env, n_stack=4)

    # Load the trained model
    model = DQN.load(args.model, env=env)
    print(f"Loaded model from {args.model}")

    for episode in range(1, args.episodes + 1):
        obs = env.reset()
        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            # deterministic=True -> greedy action selection (argmax Q-value)
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = env.step(action)
            total_reward += float(reward[0])
            steps += 1
            env.render()  # display the game window

        print(f"Episode {episode}: reward = {total_reward:.1f}, length = {steps} steps")

    env.close()


if __name__ == "__main__":
    main()


Writing play.py
Writing play.py


In [7]:
!cp train.py play.py /content/drive/MyDrive/dqn_breakout_run/artifacts/
print("Scripts backed up to Drive")

Scripts backed up to Drive
Scripts backed up to Drive


## 4 · The resilient experiment engine

All training runs **in-process** through one function with: direct-to-Drive saving, a mid-run
checkpoint callback, explicit memory cleanup between runs, per-experiment error isolation,
and DONE-flag resume logic.

In [8]:
import os, gc, json, time, traceback
import numpy as np
import torch
import ale_py, gymnasium as gym
gym.register_envs(ale_py)

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

DRIVE_DIR = "/content/drive/MyDrive/dqn_breakout_run"
ENV_ID    = "ALE/Breakout-v5"
TIMESTEPS = 150_000
N_ENVS    = 4

def log_line(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(f"{DRIVE_DIR}/run_log.txt", "a") as f:
        f.write(line + "\n")

def run_experiment(name, policy="CnnPolicy", lr=1e-4, gamma=0.99, batch=32,
                   eps_start=1.0, eps_end=0.01, eps_decay=0.10,
                   timesteps=TIMESTEPS, seed=42):
    """Train one experiment, saving models/logs DIRECTLY to Drive. Resumable via DONE flag."""
    flag = f"{DRIVE_DIR}/flags/{name}.DONE"
    if os.path.exists(flag):
        log_line(f"SKIP {name} (already completed)")
        return True

    model_dir = f"{DRIVE_DIR}/models/{name}"
    log_dir   = f"{DRIVE_DIR}/logs/{name}"
    os.makedirs(model_dir, exist_ok=True); os.makedirs(log_dir, exist_ok=True)

    env = eval_env = model = None
    try:
        log_line(f"START {name}: {policy} lr={lr} gamma={gamma} batch={batch} "
                 f"eps {eps_start}->{eps_end}@{eps_decay} steps={timesteps}")

        env      = VecFrameStack(make_atari_env(ENV_ID, n_envs=N_ENVS, seed=seed), n_stack=4)
        eval_env = VecFrameStack(make_atari_env(ENV_ID, n_envs=1, seed=seed+100), n_stack=4)

        model = DQN(policy, env,
                    learning_rate=lr, gamma=gamma, batch_size=batch,
                    buffer_size=50_000, learning_starts=20_000,
                    target_update_interval=5_000, train_freq=4,
                    exploration_initial_eps=eps_start,
                    exploration_final_eps=eps_end,
                    exploration_fraction=eps_decay,
                    verbose=0, seed=seed)

        eval_cb = EvalCallback(eval_env, best_model_save_path=model_dir, log_path=log_dir,
                               eval_freq=max(50_000 // N_ENVS, 1), n_eval_episodes=3,
                               deterministic=True, render=False)
        ckpt_cb = CheckpointCallback(save_freq=max(50_000 // N_ENVS, 1),
                                     save_path=model_dir, name_prefix="ckpt")

        t0 = time.time()
        model.learn(total_timesteps=timesteps, callback=[eval_cb, ckpt_cb], progress_bar=True)
        model.save(f"{model_dir}/dqn_model.zip")

        # summary metrics -> Drive
        d = np.load(f"{log_dir}/evaluations.npz")
        r = d["results"].mean(axis=1)
        summary = {"name": name, "policy": policy, "lr": lr, "gamma": gamma, "batch": batch,
                   "eps": [eps_start, eps_end, eps_decay], "timesteps": timesteps,
                   "best_eval_reward": float(r.max()),
                   "best_at_step": int(d["timesteps"][int(r.argmax())]),
                   "final_eval_reward": float(r[-1]),
                   "final_ep_length": float(d["ep_lengths"].mean(axis=1)[-1]),
                   "minutes": round((time.time()-t0)/60, 1)}
        with open(f"{model_dir}/summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        open(flag, "w").write("done")
        log_line(f"DONE {name}: best={summary['best_eval_reward']:.1f} "
                 f"final={summary['final_eval_reward']:.1f} ({summary['minutes']} min)")
        return True

    except Exception as e:
        log_line(f"FAILED {name}: {e}")
        with open(f"{DRIVE_DIR}/errors.txt", "a") as f:
            f.write(f"\n=== {name} ===\n{traceback.format_exc()}\n")
        return False

    finally:
        # prevent RAM/VRAM creep across experiments (typical cause of mid-run crashes)
        for obj in (model, env, eval_env):
            try: del obj
            except: pass
        gc.collect()
        torch.cuda.empty_cache()

log_line("Engine ready")

[05:40:41] Engine ready
[05:40:41] Engine ready
[05:40:41] Engine ready


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introductio

## 5 · MLP vs CNN comparison (resumable)

In [9]:
run_experiment("m0_mlp_test", policy="MlpPolicy", timesteps=50_000)
run_experiment("m0_cnn_test", policy="CnnPolicy", timesteps=50_000)

[05:40:41] START m0_mlp_test: MlpPolicy lr=0.0001 gamma=0.99 batch=32 eps 1.0->0.01@0.1 steps=50000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

## 6 · All 10 hyperparameter experiments (resumable loop)

In [10]:
EXPERIMENTS = [
  ("m2_exp01", dict()),                                                # Baseline
  ("m2_exp02", dict(lr=1e-3)),                                         # High LR
  ("m2_exp03", dict(lr=1e-5)),                                         # Low LR
  ("m2_exp04", dict(gamma=0.90)),                                      # Low gamma
  ("m2_exp05", dict(gamma=0.999)),                                     # Very high gamma
  ("m2_exp06", dict(batch=128)),                                       # Large batch
  ("m2_exp07", dict(batch=16)),                                        # Small batch
  ("m2_exp08", dict(eps_decay=0.02)),                                  # Fast eps decay
  ("m2_exp09", dict(eps_end=0.10, eps_decay=0.50)),                    # Over-exploration
  ("m2_exp10", dict(batch=64, eps_decay=0.15)),                        # Combined best
]

status = {}
for name, overrides in EXPERIMENTS:
    status[name] = run_experiment(name, **overrides)

failed = [n for n, ok in status.items() if not ok]
if failed:
    log_line(f"Completed with failures: {failed} -- rerun this cell to retry them "
             f"(completed ones are skipped automatically). See errors.txt on Drive.")
else:
    log_line("All 10 experiments completed successfully.")

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 149,992/150,000  [ 0:10:36 < 0:00:01 , 254 it/s ]

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 149,992/150,000  [ 0:10:37 < 0:00:01 , 254 it/s ]

Eval num_timesteps=150000, episode_reward=10.67 +/- 0.94

Episode length: 537.00 +/- 27.65

New best mean reward!

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 149,992/150,000  [ 0:10:37 < 0:00:01 , 254 it/s ]

[06:31:15] DONE m2_exp04: best=10.7 final=10.7 (10.6 min)
[06:31:15] START m2_exp05: CnnPolicy lr=0.0001 gamma=0.999 batch=32 eps 1.0->0.01@0.1 steps=150000


## 7 · Results table + report → Drive

In [11]:
import json, glob, os
DRIVE_DIR = "/content/drive/MyDrive/dqn_breakout_run"

labels = {
 "m2_exp01":"Baseline","m2_exp02":"High LR","m2_exp03":"Low LR","m2_exp04":"Low gamma",
 "m2_exp05":"Very high gamma","m2_exp06":"Large batch","m2_exp07":"Small batch",
 "m2_exp08":"Fast eps decay","m2_exp09":"Over-exploration","m2_exp10":"Combined best"}

rows, results = [], {}
for i,(name,label) in enumerate(labels.items(),1):
    p = f"{DRIVE_DIR}/models/{name}/summary.json"
    if not os.path.exists(p):
        rows.append(f"| {i} | {name} | RUN MISSING — rerun Section 6 |"); continue
    s = json.load(open(p))
    hp = (f"lr={s['lr']}, gamma={s['gamma']}, batch={s['batch']}, "
          f"eps_start={s['eps'][0]}, eps_end={s['eps'][1]}, eps_decay={s['eps'][2]}")
    results[name] = s["best_eval_reward"]
    rows.append(f"| {i} | {hp} | {label}: best eval reward {s['best_eval_reward']:.1f} "
                f"at {s['best_at_step']//1000}k; final {s['final_eval_reward']:.1f}; "
                f"final ep length {s['final_ep_length']:.0f} |")

table = "| # | Hyperparameter Set | Noted Behavior |\n|---|---|---|\n" + "\n".join(rows)
print(table)

best_exp = max(results, key=results.get)
report = (f"# Member 2 (Kelvin) — Hyperparameter Tuning Results\n\n{table}\n\n"
          f"**Best experiment:** {best_exp} (eval reward {results[best_exp]:.1f}). "
          f"Its best_model.zip is the submitted dqn_model.zip.\n")
with open(f"{DRIVE_DIR}/artifacts/member2_results.md","w") as f: f.write(report)

import shutil
shutil.copy(f"{DRIVE_DIR}/models/{best_exp}/best_model.zip",
            f"{DRIVE_DIR}/artifacts/dqn_model.zip")
print(f"\nBest: {best_exp} -> artifacts/dqn_model.zip | report -> artifacts/member2_results.md")

| # | Hyperparameter Set | Noted Behavior |
|---|---|---|
| 1 | lr=0.0001, gamma=0.99, batch=32, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | Baseline: best eval reward 9.7 at 150k; final 9.7; final ep length 456 |
| 2 | lr=0.001, gamma=0.99, batch=32, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | High LR: best eval reward 14.7 at 150k; final 14.7; final ep length 625 |
| 3 | lr=1e-05, gamma=0.99, batch=32, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | Low LR: best eval reward 3.0 at 100k; final 2.0; final ep length 239 |
| 4 | lr=0.0001, gamma=0.9, batch=32, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | Low gamma: best eval reward 10.7 at 150k; final 10.7; final ep length 537 |
| 5 | lr=0.0001, gamma=0.999, batch=32, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | Very high gamma: best eval reward 12.7 at 150k; final 12.7; final ep length 526 |
| 6 | lr=0.0001, gamma=0.99, batch=128, eps_start=1.0, eps_end=0.01, eps_decay=0.1 | Large batch: best eval reward 12.3 at 100k; final 11.3; final e

## 8 · Gameplay video (greedy policy) → Drive

In [12]:
import imageio
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

env = VecFrameStack(make_atari_env("ALE/Breakout-v5", n_envs=1, seed=0,
                    env_kwargs={"render_mode":"rgb_array"}), n_stack=4)
model = DQN.load(f"{DRIVE_DIR}/artifacts/dqn_model.zip", env=env)

frames, obs, ep, total, steps = [], env.reset(), 1, 0.0, 0
while ep <= 3:
    a,_ = model.predict(obs, deterministic=True)
    obs, r, done, _ = env.step(a)
    total += float(r[0]); steps += 1
    frames.append(env.render(mode="rgb_array"))
    if done.any():
        print(f"Episode {ep}: reward={total:.0f}, length={steps}")
        ep += 1; total, steps = 0.0, 0; obs = env.reset()

imageio.mimsave(f"{DRIVE_DIR}/artifacts/gameplay.mp4", frames, fps=30)
print("gameplay.mp4 saved to Drive artifacts/")

Episode 1: reward=5, length=51
Episode 2: reward=2, length=20
Episode 3: reward=0, length=2


gameplay.mp4 saved to Drive artifacts/


## 9 · Final inventory
Everything is in `MyDrive/dqn_breakout_run/`:
- `artifacts/` — train.py, play.py, **dqn_model.zip**, **gameplay.mp4**, **member2_results.md**
- `models/<exp>/` — final + best + mid-run checkpoints + summary.json per experiment
- `logs/<exp>/` — evaluations.npz (eval evidence)
- `run_log.txt`, `errors.txt` — full audit trail

If anything shows RUN MISSING or FAILED: just rerun Section 6 — completed experiments are skipped, only the missing ones retry.

In [ ]:
!ls -R /content/drive/MyDrive/dqn_breakout_run/artifacts
!tail -20 /content/drive/MyDrive/dqn_breakout_run/run_log.txt

/content/drive/MyDrive/dqn_breakout_run/artifacts:
dqn_model.zip  gameplay.mp4  member2_results.md  play.py  train.py
[05:58:45] DONE m2_exp01: best=9.7 final=9.7 (10.5 min)
[05:58:45] START m2_exp02: CnnPolicy lr=0.001 gamma=0.99 batch=32 eps 1.0->0.01@0.1 steps=150000
[06:09:10] DONE m2_exp02: best=14.7 final=14.7 (10.4 min)
[06:09:10] START m2_exp03: CnnPolicy lr=1e-05 gamma=0.99 batch=32 eps 1.0->0.01@0.1 steps=150000
[06:20:35] DONE m2_exp03: best=3.0 final=2.0 (11.4 min)
[06:20:36] START m2_exp04: CnnPolicy lr=0.0001 gamma=0.9 batch=32 eps 1.0->0.01@0.1 steps=150000
[06:31:15] DONE m2_exp04: best=10.7 final=10.7 (10.6 min)
[06:31:15] START m2_exp05: CnnPolicy lr=0.0001 gamma=0.999 batch=32 eps 1.0->0.01@0.1 steps=150000
[06:41:54] DONE m2_exp05: best=12.7 final=12.7 (10.6 min)
[06:41:55] START m2_exp06: CnnPolicy lr=0.0001 gamma=0.99 batch=128 eps 1.0->0.01@0.1 steps=150000
[06:52:43] DONE m2_exp06: best=12.3 final=11.3 (10.8 min)
[06:52:44] START m2_exp07: CnnPolicy lr=0.0001 ga